<a href="https://colab.research.google.com/github/rafiul254/Flyrank_AI-ML-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rafiul254/Flyrank_AI-ML-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 1. My lane as an ML task (type)

**Lane:** Content Refresh / Opportunity Scoring (Lane 2)

**Task type: Scoring**

This is a scoring problem. For each content page, the model produces a
continuous number — an *opportunity score* — that measures how much refresh
potential that page has right now.

It is not:
- **Classification** — I am not predicting a binary "refresh yes/no" label
- **Ranking** — I am not ordering pages against each other relative to a query
- **Clustering** — I am not grouping pages by similarity

It is **scoring** because the output is a single number per page, pages can
be compared by that number, and the score drives a prioritisation decision:
which pages should the content team refresh first?

In [7]:
task_type     = "scoring"
entity        = "content page (URL)"
output_format = "continuous opportunity score, one value per page"

print(f"Task type  : {task_type}")
print(f"Entity     : {entity}")
print(f"Output     : {output_format}")

Task type  : scoring
Entity     : content page (URL)
Output     : continuous opportunity score, one value per page


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

**What I would predict:** An *opportunity score* — a continuous proxy that
combines multiple signals to estimate how much a page would benefit from refresh.

**Where the label comes from:** This is a **defined proxy**, not an observed
outcome. There is no historical label that directly says "this page was
refreshed and traffic improved." Instead, I construct a proxy from signals
that are measurable today:

| Signal | What it captures |
|---|---|
| Traffic decline (%) | Pages losing clicks over last 90 days vs prior 90 days |
| Impression-to-click gap | High impressions, low CTR → keyword opportunity exists but content is not converting |
| Average position | Position 5–20 = "near-miss zone" — small improvement could reach page 1 |
| Content age (days) | Older pages are more likely to carry stale information |

**Proxy formula sketch:**
`opportunity_score = w1×traffic_decline + w2×impression_gap + w3×position_gap + w4×age`

Weights are tunable. The key point is that no single signal is sufficient —
the proxy must combine all of them.

**Careful language:** This is a *directional proxy*, not a ground-truth
outcome. It supports a content decision; it does not guarantee a traffic lift.

In [9]:
import pandas as pd

proxy_signals = {
    "traffic_decline_pct" : "% drop in clicks (last 90 days vs prior 90 days)",
    "impression_ctr_gap"  : "impressions high but CTR below lane average",
    "avg_position"        : "avg position between 5 and 20 (near-miss zone)",
    "content_age_days"    : "days since URL first appeared in the dataset",
}

proxy_df = pd.DataFrame(
    list(proxy_signals.items()),
    columns=["Signal", "What it captures"]
)

print("Proxy signals that compose opportunity_score:\n")
print(proxy_df.to_string(index=False))

Proxy signals that compose opportunity_score:

             Signal                                 What it captures
traffic_decline_pct % drop in clicks (last 90 days vs prior 90 days)
 impression_ctr_gap      impressions high but CTR below lane average
       avg_position   avg position between 5 and 20 (near-miss zone)
   content_age_days     days since URL first appeared in the dataset


## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

**Primary metric: Precision@K (K = 10)**

Among the top 10 pages the model flags as high-opportunity, how many show a
measurable improvement (clicks, impressions, or position) after a refresh is applied?

- **Target: Precision@10 ≥ 0.70** — at least 7 of 10 flagged pages respond positively

**Why this metric?**
The content team can only act on a small number of recommendations at a time.
It does not matter if the model scores 10,000 pages well overall — what matters
is whether the *top recommendations* are actionable and accurate.
Precision@K directly measures that business value.

**Secondary metric: Spearman rank correlation (ρ)**
Measures whether the model's ordering of pages aligns with observed outcomes.
Target: ρ > 0.60

**What "good" looks like in practice:** A content editor acts on the top 10
recommendations. At least 7 of those pages show a directional improvement
in clicks or impressions within 30 days of the refresh going live.

In [11]:
metric_config = {
    "primary_metric"    : "Precision@K",
    "K"                 : 10,
    "target"            : ">= 0.70  (7 of 10 recommended refreshes show improvement)",
    "secondary_metric"  : "Spearman rank correlation (rho)",
    "secondary_target"  : "> 0.60",
    "measurement_window": "30 days post-refresh",
}

print("Success metric configuration:\n")
for key, val in metric_config.items():
    print(f"  {key:<22}: {val}")

Success metric configuration:

  primary_metric        : Precision@K
  K                     : 10
  target                : >= 0.70  (7 of 10 recommended refreshes show improvement)
  secondary_metric      : Spearman rank correlation (rho)
  secondary_target      : > 0.60
  measurement_window    : 30 days post-refresh


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

**One row = one URL (content page)**

Each row represents a single page, aggregated across all dates in the dataset.
The columns capture the raw signals I need to compute an opportunity score.

In [13]:
import pandas as pd

csv_path = f"{REPO_PATH}/data/raw/content_refresh_anonymized.csv"
df_raw = pd.read_csv(csv_path)

unit_df = df_raw[[
    "content_id",
    "content_age_days",
    "avg_position",
    "ctr",
    "impressions_90d",
    "clicks_90d",
    "clicks_last_30d",
    "clicks_prev_30d",
    "trend_direction",
    "trend_pct",
]].copy()

unit_df["opportunity_score"] = (
    (unit_df["trend_pct"].clip(upper=0).abs() / 100) * 0.4 +   # traffic decline
    ((unit_df["avg_position"] - 1).clip(0, 30) / 30)   * 0.3 + # position gap
    ((1 - unit_df["ctr"].clip(0, 1)))                   * 0.2 + # low CTR
    (unit_df["content_age_days"].clip(0, 730) / 730)    * 0.1   # age
).round(4)

print(f"Unit of analysis : 1 row = 1 content page (content_id)")
print(f"Shape            : {unit_df.shape[0]} pages × {unit_df.shape[1]} columns")
print(f"\nTop 10 highest opportunity pages:\n")
unit_df.sort_values("opportunity_score", ascending=False).head(10)

Unit of analysis : 1 row = 1 content page (content_id)
Shape            : 30000 pages × 11 columns

Top 10 highest opportunity pages:



,content_id,content_age_days,avg_position,ctr,impressions_90d,clicks_90d,clicks_last_30d,clicks_prev_30d,trend_direction,trend_pct,opportunity_score
8407,content_d1e915d03c28,537,45.0,0.0,2,0,0,0,down,-100.0,0.9736
17896,content_b2f81797c4af,517,63.8,0.0,5,0,0,0,down,-100.0,0.9708
22875,content_4d46fbee28b6,517,31.6,0.0,52,0,0,0,down,-100.0,0.9708
11123,content_9d1cd3c0c79f,517,31.2,0.0,73,0,0,0,down,-100.0,0.9708
1371,content_25e10aa9dced,517,51.5,0.0,2,0,0,0,down,-100.0,0.9708
12977,content_930c7a86e456,517,80.2,0.0,108,0,0,0,down,-100.0,0.9708
23815,content_df23b4bc766d,502,49.6,0.0,665,0,0,0,down,-100.0,0.9688
7769,content_9524a5115a30,502,33.7,0.0,15,0,0,0,down,-100.0,0.9688
17690,content_c268b1716236,502,41.7,0.0,3,0,0,0,down,-100.0,0.9688
23523,content_38b9529bf1b6,502,32.7,0.0,36,0,0,0,down,-100.0,0.9688


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

## 5. Why ML beats a fixed rule here

**The naive fixed rule:** "Refresh any page older than 6 months."

**Why that fails:**
- A 2-year-old page with growing traffic does not need a refresh — it is working
- A 3-month-old page with rapidly declining clicks might be urgent today
- Two pages with identical age can have completely different opportunity levels

**What makes the pattern too complex for an if-statement:**

1. **Signal interactions** — A page with high impressions + low CTR + position 8
   is very different from high impressions + low CTR + position 45.
   A fixed rule cannot capture this interaction.

2. **Non-linear thresholds** — A "good" CTR depends on topic, position, and
   content type. There is no single number that works across all pages.

3. **Trade-offs between signals** — A page might be old but still ranking well,
   or new but already declining fast. The right priority depends on the
   *combination* of signals, not any one factor alone.

**What a scoring model does instead:** It learns the relative weight of each
signal from the data, captures interactions, and outputs a score that reflects
combined opportunity — something no static rule can do accurately at scale.

**Careful language:** The model provides *decision support*, not a guaranteed
outcome. A high score means "strong candidate for refresh" — not a promise
that refresh will improve traffic.

In [15]:
import pandas as pd

example = pd.DataFrame({
    "page"          : ["page_A (old but growing)", "page_B (old and declining)"],
    "age_days"      : [180,                         180],
    "clicks_trend"  : ["+14% (growing)",            "-48% (declining)"],
    "avg_position"  : [2.9,                          12.3],
    "ctr"           : [0.087,                        0.019],
    "refresh_needed": ["No",                          "Yes"],
})

print("Same age — completely different opportunity:\n")
print(example.to_string(index=False))
print()
print("A fixed rule 'refresh if age > 180 days' treats these identically.")
print("A scoring model sees the difference in clicks_trend, position, and CTR.")

Same age — completely different opportunity:

                      page  age_days     clicks_trend  avg_position   ctr refresh_needed
  page_A (old but growing)       180   +14% (growing)           2.9 0.087             No
page_B (old and declining)       180 -48% (declining)          12.3 0.019            Yes

A fixed rule 'refresh if age > 180 days' treats these identically.
A scoring model sees the difference in clicks_trend, position, and CTR.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.